# Datenerfassung

Dieses Notebook verwendet SQL.



In [7]:
USE Flugbuchung;
GO

BEGIN TRY
    BEGIN TRANSACTION;

    -- PERSON: Marcel Pott (der Kunde selbst), Petra Pott, Inge Pott (die Passagiere)
    INSERT INTO PERSON (Vorname, Nachname, Geburtsdatum)
    VALUES ('Marcel', 'Pott', NULL); -- Geburtsdatum steht nicht auf dem Beleg
    DECLARE @PersonID_Marcel INT = SCOPE_IDENTITY();

    INSERT INTO PERSON (Vorname, Nachname, Geburtsdatum)
    VALUES ('Petra', 'Pott', '1968-07-23');
    DECLARE @PersonID_Petra INT = SCOPE_IDENTITY();

    INSERT INTO PERSON (Vorname, Nachname, Geburtsdatum)
    VALUES ('Inge', 'Pott', '1988-10-31');
    DECLARE @PersonID_Inge INT = SCOPE_IDENTITY();

    -- KUNDE
    INSERT INTO KUNDE (Kundennummer, Strasse, PLZ, Ort, PersonID)
    VALUES ('00456', 'Sandweg 2', '12345', 'Berlin', @PersonID_Marcel);

    -- FLUGHAFEN
    INSERT INTO FLUGHAFEN (FlughafenCode, FlughafenName)
    VALUES ('CGN', 'Köln-Bonn'),
           ('PMI', 'Palma de Mallorca');

    -- FLUGVERBINDUNG
    INSERT INTO FLUGVERBINDUNG (Flugnummer, Abflugzeit, Ankunftzeit, Preis, AbflughafenCode, ZielflughafenCode)
    VALUES ('ET9250', '11:30', '13:30', 150.00, 'CGN', 'PMI'),
           ('ET9251', '20:20', '22:40', 150.00, 'PMI', 'CGN');

    -- VERKEHRSTAG
    INSERT INTO VERKEHRSTAG (Datum)
    VALUES ('2006-12-02'),
           ('2006-12-06');

    -- FLUG
    INSERT INTO FLUG (Flugnummer, Datum)
    VALUES ('ET9250', '2006-12-02'),
           ('ET9251', '2006-12-06');

    -- BUCHUNG
    DECLARE @BuchungID BIGINT = 3897212633; -- reale Nummer vom Beleg

    INSERT INTO BUCHUNG (BuchungID, Buchungsdatum, Kundennummer)
    VALUES (@BuchungID, '2006-11-01', '00456'); -- Buchungsdatum steht nicht auf dem Beleg

    -- BUCHUNGSPOSTEN: Petra und Inge fliegen jeweils Hin- und Rückflug
    INSERT INTO BUCHUNGSPOSTEN (BuchungID, PersonID, Flugnummer, Datum)
    VALUES (@BuchungID, @PersonID_Petra, 'ET9250', '2006-12-02'),
           (@BuchungID, @PersonID_Petra, 'ET9251', '2006-12-06'),
           (@BuchungID, @PersonID_Inge,  'ET9250', '2006-12-02'),
           (@BuchungID, @PersonID_Inge,  'ET9251', '2006-12-06');

    COMMIT TRANSACTION;
    PRINT 'Alle Daten erfolgreich eingefügt.';
END TRY
BEGIN CATCH
    IF @@TRANCOUNT > 0
        ROLLBACK TRANSACTION;
    PRINT 'Fehler beim Einfügen: ' + ERROR_MESSAGE();
END CATCH
GO

Commands completed successfully.

(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)
(2 rows affected)
(2 rows affected)
(2 rows affected)
(2 rows affected)
(1 row affected)
(4 rows affected)
Alle Daten erfolgreich eingefügt.

Total execution time: 00:00:00.057

In [11]:
CREATE OR ALTER VIEW vw_Buchungsdetails
AS
SELECT
    b.BuchungID,
    b.Buchungsdatum,
    k.Kundennummer,
    kp.Vorname   AS KundenVorname,
    kp.Nachname  AS KundenNachname,
    p.Vorname    AS PassagierVorname,
    p.Nachname   AS PassagierNachname,
    p.Geburtsdatum,
    bp.Flugnummer,
    bp.Datum     AS Flugdatum,
    fv.Abflugzeit,
    fv.Ankunftzeit,
    fv.Preis,
    fh_ab.FlughafenName AS Abflughafen,
    fh_zu.FlughafenName AS Zielflughafen
FROM BUCHUNG b
JOIN KUNDE k            ON b.Kundennummer = k.Kundennummer
JOIN PERSON kp           ON k.PersonID = kp.PersonID
JOIN BUCHUNGSPOSTEN bp   ON b.BuchungID = bp.BuchungID
JOIN PERSON p             ON bp.PersonID = p.PersonID
JOIN FLUG f               ON bp.Flugnummer = f.Flugnummer AND bp.Datum = f.Datum
JOIN FLUGVERBINDUNG fv    ON f.Flugnummer = fv.Flugnummer
JOIN FLUGHAFEN fh_ab      ON fv.AbflughafenCode = fh_ab.FlughafenCode
JOIN FLUGHAFEN fh_zu      ON fv.ZielflughafenCode = fh_zu.FlughafenCode;
GO

Commands completed successfully.

Total execution time: 00:00:00.029

In [14]:
SELECT * FROM vw_Buchungsdetails
ORDER BY Flugdatum, PassagierNachname, PassagierVorname;

(4 rows affected)

BuchungID  | Buchungsdatum | Kundennummer | KundenVorname | KundenNachname | PassagierVorname | PassagierNachname | Geburtsdatum | Flugnummer | Flugdatum  | Abflugzeit | Ankunftzeit | Preis  | Abflughafen       | Zielflughafen    
-----------+---------------+--------------+---------------+----------------+------------------+-------------------+--------------+------------+------------+------------+-------------+--------+-------------------+------------------
3897212633 | 2006-11-01    | 00456        | Marcel        | Pott           | Inge             | Pott              | 1988-10-31   | ET9250     | 2006-12-02 | 11:30:00   | 13:30:00    | 150.00 | Köln-Bonn         | Palma de Mallorca
3897212633 | 2006-11-01    | 00456        | Marcel        | Pott           | Petra            | Pott              | 1968-07-23   | ET9250     | 2006-12-02 | 11:30:00   | 13:30:00    | 150.00 | Köln-Bonn         | Palma de Mallorca
3897212633 | 2006-11-01    | 00456        | Marcel       